# Gradient Descent

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from course_utils.plots import loss_landscape, plot_scatter_plot, mark_gradient_descent_path

In the previous lesson, we learned how to fit a linear regression model using the **Normal Equation**. This provides a closed-form solution, allowing us to calculate the optimal model parameters directly without searching for them iteratively.

Unfortunately, this approach does not generalise to many of the models used in modern machine learning. For example, neural networks do not have a closed-form solution, meaning there is no equation that can be solved directly to obtain the optimal parameters.

We therefore need a more general optimisation technique. **Gradient Descent** is an iterative algorithm that can minimise a loss function even when no closed-form solution exists.

Although gradient descent is not required to fit a linear regression model, it is one of the most important optimisation algorithms in machine learning. We will therefore introduce it now using linear regression as a simple example before applying it to more complex models where no closed-form solution exists.

## Create Example Observation and Loss Function

As we did in the previous lesson we will start with an example observation and a loss landscape.

Let's look at an example observation of heights (cm) and weights (kg).

In [ ]:
heights_cm = np.array([
    172, 158, 184, 166, 177,
    191, 163, 180, 155, 174,
    188, 169, 182, 160, 176,
    193, 167, 185, 171, 179
], dtype=float)

weights_kg = np.array([
    74, 61, 92, 68, 83,
    106, 57, 78, 52, 89,
    84, 98, 76, 64, 71,
    112, 73, 87, 69, 95
], dtype=float)

plot_scatter_plot(
    x=heights_cm,
    y=weights_kg,
    x_label="Height (cm)",
    y_label="Weight (kg)"
)

As in previous examples we will use the ***Mean Square Error (MSE)*** for our loss functon here which we define as:

In [ ]:
def mse(y_observed: np.array, y_predicted: np.array) -> float:
    return np.mean((y_observed - y_predicted) ** 2)

## Visualise Possible Loss Values

Before we look at how to find the minimum possible loss value let's assume that the model we want to fit to our observations is a single input linear regression model of the form:

$$
\operatorname{weight} = m \times \operatorname{height} + b
$$

where:

- $m$ is the gradient,
- $b$ is the intercept.

Now lets define a reasonable set of possible values for $m$ and $b$ respectively and plot them against the result of the loss function for each pair of $m, b$.

We call this our ***Loss Landscape***.

In [ ]:
possible_m_values = np.linspace(0.5, 1.8, 100)
possible_b_values = np.linspace(-180, -80, 100)

fig, ax = loss_landscape(
    x_observed=heights_cm,
    y_observed=weights_kg,
    possible_m_values=possible_m_values,
    possible_b_values=possible_b_values,
    loss_function=mse)

## Gradient Descent

**Gradient Descent** is an iterative optimisation algorithm that minimises a function by repeatedly taking small steps in the direction opposite to its gradient.

## Hyperparameters

A ***hyperparameter*** is something you choose before training begins that controls how the model is trained. 

This is opposed to a ***parameter*** which is what the model learns from the data. Here our parameters are the gradient ($m$) and intercept($b$).

To control this process we introduce two hyperparameters:

* The **learning rate**, which determines the size of each step taken.
* The **maximum number of epochs**, which determines the maximum number of optimisation steps the algorithm is allowed to take.

Both hyperparameters involve trade-offs.

* If the learning rate is **too small**, progress towards the minimum is slow and many epochs may be required before the algorithm converges. 
* If the learning rate is **too large**, the algorithm may repeatedly overshoot the minimum, causing the loss to oscillate or even diverge.

An **epoch** is one complete optimisation step. During each epoch, every model parameter is updated once. For our linear regression model, this means both the gradient ($m$) and intercept ($b$) are updated during each epoch.

The maximum number of epochs places an upper bound on the amount of computation. 
* If too few epochs are allowed, training may stop before the model has reached a good solution, leaving the loss unnecessarily high.
* If too many epochs are allowed, the algorithm will continue updating after it has effectively converged. For simple models such as linear regression this usually just wastes computation, while for more complex models it can eventually lead to overfitting.

## Implementing the Gradient Descent Algorithm

Below we shall implement the gradient descent algorithm as a Python function:

In [ ]:
def perform_gradient_descent(
    X_observed: np.ndarray,
    y_observed: np.ndarray,
    initial_parameters: np.ndarray,
    learning_rate: float,
    epochs: int,
) -> tuple[np.ndarray, float, list[np.ndarray], list[float]]:
    parameter_history = []
    loss_history = []

    X_observed = np.asarray(X_observed, dtype=float)
    if X_observed.ndim == 1:
        X_observed = X_observed.reshape(-1, 1)

    y_observed = np.asarray(y_observed, dtype=float).reshape(-1)

    # Create design matrix by adding intercept column
    X = np.column_stack([
        X_observed,
        np.ones(len(X_observed)),
    ])

    current_parameters = np.asarray(initial_parameters, dtype=float).copy()
    n = len(X)

    for epoch in range(epochs):
        # Predict y values using current parameters
        y_pred = X @ current_parameters

        # Compute loss using the notebook's MSE function
        loss = mse(y_observed, y_pred)

        parameter_history.append(current_parameters.copy())
        loss_history.append(loss)

        # Compute the gradients of MSE with respect to each parameter
        gradients = (-2 / n) * X.T @ (y_observed - y_pred)

        # Update the parameters using the gradients (we subtract so that the direction is the opposite to that of the gradient)
        current_parameters -= learning_rate * gradients

    # Set the final parameters and final loss value
    final_parameters = current_parameters
    final_loss = mse(y_observed, X @ final_parameters)

    return (final_parameters, final_loss, parameter_history, loss_history)

#### Note on How the Gradients Were Calculated Above

To calculate the gradient, we first differentiate the loss function with respect to each model parameter. 

We begin with the **Mean Squared Error (MSE)** loss function for a linear regression model, which we can rewrite this as a vector multiplication:

$$
\begin{aligned}
J(\theta)
&=
\frac{1}{n}
\sum_{i=1}^{n}
\left(y_i-\hat{y}_i\right)^2 \\

&=
\frac{1}{n}
\left(\mathbf y-\hat{\mathbf y}\right)^T
\left(\mathbf y-\hat{\mathbf y}\right) \\

&=
\frac{1}{n}
\left(\mathbf y-X\theta\right)^T
\left(\mathbf y-X\theta\right).
\end{aligned}
$$

where $n$ is the number of observations, $X$ is the ***Design Matrix***, $p$ is the number of weights and $\theta$ is the vector of parameters such that

$$
X 
= 
\begin{bmatrix}
x_{11} & x_{12} & \cdots & x_{1p} & 1 \\
x_{21} & x_{22} & \cdots & x_{2p} & 1 \\
\vdots & \vdots & \ddots & \vdots & \vdots \\
x_{n1} & x_{n2} & \cdots & x_{np} & 1
\end{bmatrix}
$$

and

$$
\theta = \begin{bmatrix} w_1 \\ w_2 \\ \dots \\ w_p \\ b \end{bmatrix}
$$

Now we have our loss function as a vector equation we can use the chain rule to differentiate with respect to $\theta$.

We will use the following subtitution to help demonstrate this:

$$
J(\theta)
=
\frac{1}{n}
r^Tr
$$

where $r = \mathbf y-X\theta$. 

The chain rule for differentiation is as follows:

$$
\frac{d}{dx}f(g(x))
=
f'(g(x))\,g'(x)
$$

So we have that:

$$
\begin{aligned}
\frac{\partial}{\partial\theta} J(\theta)
&=
\frac{\partial}{\partial\theta}\left(\frac{1}{n}r^Tr\right) \\[6pt]
&=
\frac{1}{n}\frac{\partial}{\partial\theta}\left(r^Tr\right) \\[6pt]
&=
\frac{1}{n}
\left(
\frac{\partial r}{\partial \theta}
\right)^T
\frac{d}{dr}
\left(r^Tr\right)
\\[6pt]
&=
\frac{1}{n}
\left(
\frac{\partial}{\partial \theta}\left(y - X\theta\right)
\right)^T
\frac{d}{dr}
\left(r^Tr\right)
\\[6pt]
&=
\frac{1}{n}
\left(-X\right)^T
(2r)
\\[6pt]
&=
-\frac{2}{n}
X^Tr
\\[6pt]
&=
-\frac{2}{n}
X^T
(\mathbf y-X\theta)
\end{aligned}
$$

Therefore to find the gradients of our loss function for a given $\theta$ over a set of observations we have the following:

$$
\begin{aligned}
\nabla_\theta J(\theta)
&=
-\frac{2}{n}
X^T
\left(\mathbf y-X\theta\right) \\[6pt]
&=
-\frac{2}{n}
X^T
\left(\mathbf y-\hat{y}\right)
\end{aligned}
$$

which is implemented in our code above as

```python
gradients = (-2 / n) * X.T @ (y_observed - y_pred)
```

## Perform Gradient Descent on Our Observations

Now have our gradient descent formula we can apply it to our observed height and weights in order to create a linear regression that can predict a weight (kg) from a given height (cm).

In [ ]:
initial_parameters = np.array([0.5, -130.0])
learning_rate = 0.000001
epochs = 5000

final_parameters, final_loss, parameter_history, loss_history = perform_gradient_descent(
    X_observed=heights_cm.reshape(-1, 1),
    y_observed=weights_kg,
    initial_parameters=initial_parameters,
    learning_rate=learning_rate,
    epochs=epochs,
)

print("Using gradient descent with the following hyperparameters:")
print("Initial Parameters:")
print(f"Gradient (m): {initial_parameters[0]:.2f}")
print(f"Intercept (b): {initial_parameters[1]:.2f}")
print(f"Learning Rate: {learning_rate}")
print(f"Epochs: {epochs}")
print("We have found that the following parameter values minimise the loss when fitting our Linear Regression to our observed data:")
print(f"Gradient (m): {final_parameters[0]:.2f}")
print(f"Intercept (b): {final_parameters[1]:.2f}")
print(f"Loss: {final_loss:.2f}")

### Visualising Gradient Descent on Our Loss Landscape

Now we have performed gradient descent, and recorded the progress along the way, we can visualise this on our loss landscape.

In [ ]:
m_history = [p[0] for p in parameter_history]
b_history = [p[1] for p in parameter_history]

fig, ax = loss_landscape(
    x_observed=heights_cm,
    y_observed=weights_kg,
    possible_m_values=possible_m_values,
    possible_b_values=possible_b_values,
    loss_function=mse
)

mark_gradient_descent_path(ax, m_history, b_history, loss_history)

### Plotting the Fitted Linear Regression

Finally, we can plot our fitted linear regression against our observations to see how it fits:

In [ ]:
# Heights covering the full plotted range
line_heights = np.linspace(
    heights_cm.min(),
    heights_cm.max(),
    100
)

# Weights predicted across our full plotted range of heights
m = final_parameters[0]
b = final_parameters[1]

line_weights = m * line_heights + b

plot_scatter_plot(
    x=heights_cm, 
    y=weights_kg, 
    x_label="Height (cm)", 
    y_label="Weight (kg)",
    line_values=(line_heights, line_weights)
)
